# JRA レース予測: 1 着になる確率 (LightGBM + Optuna)

各馬が 1 着になる確率を予測し, そこから着順をシミュレーションしてワイド (2 頭とも 3 着以内) の的中確率を計算する.

- モデル: 条件付きロジット (レース内ソフトマックス). LightGBM が各馬の強さの対数 (`win_score`) を出力し,
  レース内でソフトマックスをとって 1 着の確率 (`p_win`) にする → **レースごとの合計は必ず 1**.
  学習では「1 着の馬に付けた確率の -log」(レース単位の logloss) を最小化する (LightGBM のカスタム目的関数)

- 目的変数: `is_win` (1 着なら 1. CSV の着順から作る) / 特徴量: `docs/racing_predict_model_spec.md` 参照 (オッズ・人気・馬名・着順は使わない)
- `model.market_offset: true` (`configs/train.yaml`) なら **オッズとの差だけを学習** する: 単勝オッズから逆算した確率を
  出発点 (LightGBM の init_score) にし, 木は「オッズがどれだけ外れているか」だけを学ぶ (オッズは特徴量に入れない).
  推論では発走前の単勝オッズを使う
- **メインの目標: モデルを混ぜるとオッズ単体より当たること** (= モデルにオッズにない情報があること). 4 章で評価する
- 検証は時系列 (学習 = 過去, テスト = 直近の開催日のうち全開催日数の `test_ratio` の割合). Optuna もテスト期間を使わずに時系列 CV で探索
- 学習データの期間・テストの比率・Optuna の設定は `configs/train.yaml` で指定する (このノートブックは編集不要)
- 処理の本体は `src/racing_prediction/` (前処理 `features/model_features.py`, 学習 `models/train_model.py`, 推論 `models/predict_model.py`)

## Kaggle で実行する
ローカルで次を実行すると, データ・コードのアップロード → Kaggle でこのノートブックを実行 (Save & Run All)
→ 完了待ち → 学習済みモデルを `models/production/` に取り込む, までを行う (詳細は `docs/github_actions_setup.md`).
```
python scripts/train_on_kaggle.py --update-dataset
```
評価結果 (指標・予測・グラフなど) は `results/` に保存され, スクリプトがローカルの `reports/kaggle_runs/<日時>/` にダウンロードする.
ローカルで実行する場合は準備不要 (`data/processed/races` と `src/` をそのまま使う).


In [ ]:
# autoreload はローカルで src/ を編集しながら実行するときだけ使う
# (Kaggle の古い IPython は Python 3.12 で削除された imp を使っていて読み込めないため)
from pathlib import Path

if not Path("/kaggle/input").exists():
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")

import sys
import warnings
from pathlib import Path

import pandas as pd

IS_KAGGLE = Path("/kaggle/input").exists()

def find_kaggle_inputs(input_dir: Path = Path("/kaggle/input"), work_dir: Path = Path("/kaggle/working/inputs")):
    """Kaggle Dataset として追加した src/ (racing_prediction パッケージ) と races/ (CSV) の場所を探す.

    レースの CSV は半年ごとに別のデータセット (racing-prediction-data-races-2024h1 など) に分けてあるため,
    races/ は複数見つかる. (src の親フォルダ, races/ のフォルダのリスト) を返す.

    - os.walk(followlinks=True) で探す (Path.rglob は Python 3.12 以前だと
      シンボリックリンクのフォルダの中を "**" で探さないため)
    - src.zip / races.zip のように圧縮ファイルのまま置かれている場合は work_dir に展開して使う
      (データセットごとに別のフォルダに展開する. どのデータセットにも races.zip があるため)
    """
    import os
    import zipfile

    def walk(root: Path):
        for dirpath, dirnames, filenames in os.walk(root, followlinks=True):
            yield Path(dirpath), dirnames, filenames

    def has_csv(root: Path) -> bool:
        return any(name.endswith(".csv") for _, _, filenames in walk(root) for name in filenames)

    # 圧縮ファイルのままの場合は展開する
    for dirpath, _, filenames in walk(input_dir):
        for name in filenames:
            if name.endswith(".zip"):
                dest = work_dir / dirpath.relative_to(input_dir) / Path(name).stem
                if not dest.exists():
                    with zipfile.ZipFile(dirpath / name) as zf:
                        zf.extractall(dest)

    src_dir = None
    races_dirs: list[Path] = []
    for root in (input_dir, work_dir):
        if not root.exists():
            continue
        for dirpath, dirnames, filenames in walk(root):
            if src_dir is None and dirpath.name == "racing_prediction" and "__init__.py" in filenames:
                src_dir = dirpath.parent
            if dirpath.name == "races" and not any(r in dirpath.parents for r in races_dirs) and has_csv(dirpath):
                races_dirs.append(dirpath)

    if src_dir is None or not races_dirs:
        listing = "\n".join(
            f"  {p.relative_to(input_dir)}{'/' if p.is_dir() else ''}"
            for p in sorted(input_dir.glob("*/*"))[:50]
        ) or "  (空です. データセットが 1 つも追加されていません)"
        missing = [name for name, ok in [("src/racing_prediction (学習コード)", src_dir), ("races/ (CSV)", races_dirs)] if not ok]
        raise FileNotFoundError(
            f"/kaggle/input に {', '.join(missing)} が見つかりません.\n"
            f"/kaggle/input の中身:\n{listing}\n\n"
            "対処法: ローカルで python scripts/train_on_kaggle.py --update-dataset を実行する\n"
            "  (データセットのアップロードと, このノートブックへのデータセットの追加を自動で行う)"
        )
    return src_dir, races_dirs


def check_dataset_fingerprint(expected: dict[str, str] | None, input_dir: Path = Path("/kaggle/input")) -> None:
    """使われているデータセットが, アップロードした最新版かを識別値 (<データセット名>.fingerprint.txt) で確かめる.

    Kaggle はデータセットを更新した直後だと前の版でノートブックを実行することがあり,
    そのまま進むと古いコード・設定 (configs/train.yaml)・データで学習してしまうため, ここで止める.
    """
    import os

    suffix = ".fingerprint.txt"
    found = {}
    for dirpath, _, filenames in os.walk(input_dir, followlinks=True):
        for name in filenames:
            if name.endswith(suffix):
                found[name[: -len(suffix)]] = (Path(dirpath) / name).read_text(encoding="utf-8").strip()
    print("データセットの識別値:", found)
    if expected is None:  # Kaggle の画面から手動で実行した場合は確認しない
        return
    wrong = {slug: (found.get(slug), fp) for slug, fp in expected.items() if found.get(slug) != fp}
    if wrong:
        detail = ", ".join(f"{slug} (使われている {got} / 最新版 {fp})" for slug, (got, fp) in wrong.items())
        raise RuntimeError(
            f"アップロードした最新版ではないデータセットがあります: {detail}. "
            "Kaggle への反映が間に合っていない可能性があります. 数分おいてから "
            "`python scripts/train_on_kaggle.py` を再実行してください."
        )


# scripts/train_on_kaggle.py で実行すると, アップロードしたデータセットの {データセット名: 識別値} がここに埋め込まれる
EXPECTED_DATASET_FINGERPRINT = None

if IS_KAGGLE:
    SRC_DIR, RACES_DIR = find_kaggle_inputs()   # RACES_DIR は races/ のリスト (半年ごとのデータセット)
    check_dataset_fingerprint(EXPECTED_DATASET_FINGERPRINT)
    ARTIFACTS_DIR = Path("/kaggle/working/artifacts")
    RESULTS_DIR = Path("/kaggle/working/results")   # 評価結果 (scripts/train_on_kaggle.py がローカルに保存する)
else:
    ROOT = Path.cwd().parent
    SRC_DIR = ROOT / "src"
    RACES_DIR = ROOT / "data" / "processed" / "races"
    ARTIFACTS_DIR = ROOT / "models" / "latest"
    RESULTS_DIR = ROOT / "reports" / "latest"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

# グラフで日本語を表示するフォント (Kaggle ではデータセットに同梱した assets/fonts/ の Noto Sans JP を使う).
# 見つからなければ None になり, グラフは英語で描く (文字化けさせないため)
from racing_prediction.visualization.visualize import setup_japanese_font

JAPANESE_FONT = setup_japanese_font([SRC_DIR.parent / "assets" / "fonts"])
print("グラフの日本語フォント:", JAPANESE_FONT or "なし (グラフは英語で表示)")

import optuna

from racing_prediction.features.model_features import load_race_csvs, labeled
from racing_prediction.models.train_model import (
    evaluate,
    filter_period,
    filter_venues,
    save_artifacts,
    time_split,
    train_final,
    tune,
)
from racing_prediction.models.predict_model import (
    calibration_table,
    compare_calibration,
    compare_top_pick,
    compare_with_market,
    feature_importance,
    roc_auc_summary,
    evaluate_predictions,
    load_artifacts,
    market_blend_verdict,
    odds_baseline,
    predict_win,
    simulate_top3,
    top_pick_table,
    wide_probabilities,
)

pd.set_option("display.max_columns", 50)
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ---- 設定 (configs/train.yaml から読み込む) ----
from racing_prediction.config import load_train_config

CONFIG_PATH = SRC_DIR.parent / "configs" / "train.yaml"   # Kaggle ではデータセット内の configs/train.yaml
cfg = load_train_config(CONFIG_PATH)

DATA_START_DATE = cfg.start_date   # 学習・テストに使うデータの開始日 (None なら制限なし)
DATA_END_DATE = cfg.end_date       # 学習・テストに使うデータの終了日 (None なら制限なし)
VENUES = cfg.venues                # 学習・テストに使う競馬場 (None なら全競馬場)
TEST_RATIO = cfg.test_ratio        # 全開催日数のうち, 直近の何割の開催日をテストにするか
N_TRIALS = cfg.n_trials            # Optuna の試行回数
TIMEOUT = cfg.timeout_sec          # Optuna の打ち切り時間 (秒)
N_FOLDS = cfg.n_folds              # 時系列 CV の fold 数
MARKET_OFFSET = cfg.market_offset  # True: オッズとの差だけを学習する

print("設定:", CONFIG_PATH)
for key, value in cfg.to_dict().items():
    print(f"  {key}: {value}")

print("Kaggle" if IS_KAGGLE else "ローカル", "| src:", SRC_DIR, "| races:", RACES_DIR)

## 1. データの読み込み
結果が確定したレース (`is_win` がある行) のうち, `configs/train.yaml` の `data.start_date` 〜 `data.end_date` の期間のもの
(`data.venues` を指定した場合はその競馬場のもの. `null` なら `data/processed/races` にある全競馬場) を学習・評価に使う.

In [ ]:
df_all = load_race_csvs(RACES_DIR)
df = filter_venues(filter_period(labeled(df_all), DATA_START_DATE, DATA_END_DATE), VENUES)

print(f"{df['race_id'].nunique()} レース / {len(df)} 頭 / {df['race_date'].nunique()} 開催日  ({df['race_date'].min().date()} 〜 {df['race_date'].max().date()})")
print("1着の割合:", round(df["is_win"].mean(), 4), "/ 1レースあたりの1着の頭数:", df.groupby("race_id")["is_win"].sum().value_counts().to_dict())
print("競馬場:", ", ".join(f"{v} {n} レース" for v, n in df.groupby("race_venue")["race_id"].nunique().items()))
display(df.groupby(["race_venue", "race_surface"])["race_id"].nunique().rename("レース数").unstack(fill_value=0))

In [ ]:
# 平均着順の欠損率 (出走経験がない条件は NaN. LightGBM は NaN をそのまま扱える)
avg_cols = [c for c in df.columns if c.startswith("avg_rank_")]
df[avg_cols].isna().mean().sort_values().rename("欠損率").to_frame().T

## 2. 時系列での分割
開催日を新しい順に並べ, 全開催日数の `split.test_ratio` (`configs/train.yaml`) の割合をテスト (チューニングにも使わない), それより前を学習にする.
カレンダーの月数ではなく開催日の数で分けるので, 開催がない月があってもテストの量が安定する.

In [ ]:
train, test = time_split(df, test_ratio=TEST_RATIO)
for name, d in [("学習", train), ("テスト", test)]:
    print(f"{name}: {d['race_date'].min().date()} 〜 {d['race_date'].max().date()}  {d['race_date'].nunique()} 開催日 / {d['race_id'].nunique()} レース / {len(d)} 頭")

## 3. Optuna によるハイパーパラメータ探索
学習期間内で期間をずらした時系列 CV (`tuning.n_folds` fold. 試行回数・打ち切り時間も `configs/train.yaml` の `tuning`) の平均 logloss (レース単位: 1 着の馬に付けた確率の -log の平均) を最小化する.
オッズとの差だけを学習する場合 (`MARKET_OFFSET`), この logloss はオッズを出発点にした予測のもの (オッズより小さければ, オッズより当たっている).

In [ ]:
result = tune(train, n_trials=N_TRIALS, timeout=TIMEOUT, n_folds=N_FOLDS, market_offset=MARKET_OFFSET)

print("CV logloss (レース単位):", round(result.best_score, 4))
print("木の本数:", result.num_boost_round)
result.best_params

In [ ]:
# どのハイパーパラメータが効いたか (試行が少ないと不安定)
try:
    display(pd.Series(optuna.importance.get_param_importances(result.study)).rename("重要度").to_frame())
except Exception as e:
    print("重要度を計算できませんでした:", e)

## 4. 学習期間全体で学習し, テスト期間で評価
モデルと, **確定単勝オッズから逆算した確率** (1 / オッズ をレース内の合計が 1 になるよう揃えたもの) で予想した場合を同じ指標で比べる
(オッズを使わないモデルなら, オッズだけの予想にどこまで近づけるかの目安. オッズとの差を学習したモデルなら, オッズを上回れたかがわかる).

- `roc_auc`: ROC-AUC. 1 着の馬をどれだけ上位に並べられたか (0.5 = ランダム, 1.0 = 完璧. 全レースの馬をまとめて計算)
- `roc_auc_by_race`: レースごとの ROC-AUC の平均 (同じレース内での順番付けの良さ)
- `logloss`: レース単位の logloss (1 着の馬に付けた確率の -log の平均. 確率としての正確さ. 小さいほど良い)
- `brier`: 各馬の 1 着の確率の Brier スコア (小さいほど良い)
- `win_hit`: 予測 1 位の馬が 1 着だった割合 (単勝の的中率)
- `winner_in_top3`: 1 着の馬が予測上位 3 頭に入っていた割合
- `top3_precision`: 予測上位 3 頭のうち実際に 3 着以内だった割合
- `wide_top_pair_hit`: 予測上位 2 頭のワイドが的中したレースの割合

In [ ]:
builder, booster = train_final(train, result.best_params, result.num_boost_round, market_offset=MARKET_OFFSET)
metrics, pred_test = evaluate(booster, builder, test)

# レースごとの 1 着の確率の合計 (必ず 1 になる)
race_sums = pred_test.groupby("race_id")["p_win"].sum()
print(f"レースごとの p_win の合計: 最小 {race_sums.min():.6f} / 最大 {race_sums.max():.6f}")

# 比較: 確定単勝オッズから逆算した確率だけで予想した場合 (モデルはオッズを使わない)
fav = odds_baseline(test)
baseline = evaluate_predictions(fav)

display(pd.DataFrame({"モデル": pd.Series(metrics), "参考: 単勝オッズ": pd.Series(baseline)}))
print(f"ROC-AUC: モデル {metrics['roc_auc']:.4f} (レース内平均 {metrics['roc_auc_by_race']:.4f}) "
      f"/ 単勝オッズ {baseline['roc_auc']:.4f} (レース内平均 {baseline['roc_auc_by_race']:.4f})")

# 競馬場ごとの比較 (複数の競馬場で学習した場合. テストのレースが少ない競馬場はぶれが大きい)
if pred_test["race_venue"].nunique() > 1:
    rows = {}
    for venue, part in pred_test.groupby("race_venue"):
        odds_part = fav[fav["race_venue"] == venue]
        m, o = evaluate_predictions(part), evaluate_predictions(odds_part)
        rows[venue] = {"レース数": part["race_id"].nunique(), "ROC-AUC モデル": m["roc_auc"], "ROC-AUC 単勝オッズ": o["roc_auc"],
                       "logloss モデル": m["logloss"], "logloss 単勝オッズ": o["logloss"],
                       "1着的中率 予測1位": m["win_hit"], "1着的中率 1番人気": o["win_hit"]}
    display(pd.DataFrame(rows).T.round(4))

### メインの目標: モデルを混ぜると, オッズ単体より当たるか
単勝オッズには多くの人の予想が織り込まれているので, オッズと同じ情報しか持たないモデルは役に立たない.
そこで「モデルの予測をオッズと混ぜたら, オッズ単体より当たるようになるか」(= モデルにオッズにない情報があるか) を目標にする.

- 混ぜ方: 各馬のスコア = a × log(モデルの確率) + b × log(オッズの確率) をレース内でソフトマックス (a, b は最尤推定)
- 比べる相手の「オッズ単体」は, そのままのオッズと b × log(オッズの確率) で補正したもののうち良い方 (人気薄の過大評価を補正しただけで良くなる分は, モデルの手柄にしない)
- テスト期間を前半・後半に分け, 片方で a, b を決めてもう片方を予測する (決めたのと同じレースで評価すると甘くなるため)
- 「当たる」かは logloss (1 着の馬に付けた確率の -log. 確率としての当たり具合) で判定する. **混ぜた予測かモデル単体の改善の 95% 信頼区間が 0 より上なら目標達成**
  (オッズとの差を学習したモデルは, モデル単体がすでにオッズと混ざった予測).
  1 着的中率も表示するが, 数千レースでは数 % の差が偶然でも出るため判定には使わない
- モデルの重み a: 0 より大きく, p 値が小さいほどモデルがオッズにない情報を持っている
- 期待値 (混ぜた予測の確率 × 単勝オッズ) が 1 を超える馬の単勝を 100 円ずつ買った場合の回収率 (参考)


In [ ]:
market_blend = mb = compare_with_market(pred_test)

print("【メインの目標】モデルを混ぜると, オッズ単体より当たるか")
print(f"テスト期間 {mb['n_races']} レース (前半・後半で交差して, 混ぜ方を決めていない方のレースで評価)")
display(pd.DataFrame({
    "logloss (小さいほど当たる)": [mb["market_logloss"], mb["blend_logloss"], mb["model_logloss"]],
    "1 着的中率": [mb["market_hit"], mb["blend_hit"], mb["model_hit"]],
}, index=["オッズ単体", "混ぜた予測", "モデル単体"]).style.format({"logloss (小さいほど当たる)": "{:.4f}", "1 着的中率": "{:.1%}"}))
print(f"logloss の改善 (オッズ単体 - 混ぜた予測): {mb['logloss_gain']:+.4f}  "
      f"(95% 信頼区間 {mb['logloss_gain_ci_low']:+.4f} 〜 {mb['logloss_gain_ci_high']:+.4f})")
print(f"logloss の改善 (オッズ単体 - モデル単体): {mb['model_gain']:+.4f}  "
      f"(95% 信頼区間 {mb['model_gain_ci_low']:+.4f} 〜 {mb['model_gain_ci_high']:+.4f})")
print(f"混ぜ方の重み: モデル {mb['model_weight']:.3f} / オッズ {mb['market_weight']:.3f}  "
      f"(モデルの重みが 0 より大きいかの p 値 {mb['model_weight_p_value']:.3f})")
print(f"期待値 > {mb['value_threshold']} の馬の単勝: {mb['n_value_bets']} 頭, 回収率 {mb['value_bet_return']:.1%}")
print("→", market_blend_verdict(mb))


### 予測 1 位の馬 vs 1 番人気の馬 (1 着になった割合・単勝の回収率)
レースごとに, **モデルのスコアがトップの馬** と **単勝オッズが一番低い馬 (1 番人気)** のどちらが 1 着になったかを比べる.

- 1 着的中率: それぞれの馬が 1 着になったレースの割合
- 差の 95% 信頼区間: レースを復元抽出するブートストラップで計算. 0 をまたいでいれば「偶然の範囲で, 差があるとは言えない」
- p 値: マクネマー検定 (片方だけが当たったレースの数に偏りがあるか). 0.05 未満なら差があると言える
- 同じ馬 / 違う馬: 予測 1 位と 1 番人気が同じ馬だったレースと, 違う馬だったレースに分けた的中率
  (違う馬だったレースで予測 1 位の方がよく当たるなら, モデルが人気にない情報を捉えている)
- 単勝 100 円ずつ: 各レースで予測 1 位 / 1 番人気の単勝を 100 円ずつ買った場合の総利益と回収率 (払戻の合計 / 賭け金の合計).
  払戻は確定した単勝オッズ x 100 円. 回収率 100% で元が取れる (JRA の単勝の控除率は 20% なので, 全体の平均は約 80%)


In [ ]:
top_pick = compare_top_pick(pred_test)
picks = top_pick_table(pred_test)

print(f"テスト期間 {top_pick['n_races']} レース")
print(f"  予測 1 位の馬が 1 着: {top_pick['model_hit']:.1%}")
print(f"  1 番人気の馬が 1 着: {top_pick['favorite_hit']:.1%}")
print(f"  差 (予測 1 位 - 1 番人気): {top_pick['diff']:+.1%}  "
      f"(95% 信頼区間 {top_pick['diff_ci_low']:+.1%} 〜 {top_pick['diff_ci_high']:+.1%}, p 値 {top_pick['p_value']:.3f})")
verdict = ("予測 1 位の方がよく当たる" if top_pick["diff_ci_low"] > 0 else
           "1 番人気の方がよく当たる" if top_pick["diff_ci_high"] < 0 else "差があるとは言えない")
print(f"  → {verdict}")

bet_total = top_pick["bet_per_race"] * top_pick["n_races"]
print()
print(f"単勝 {top_pick['bet_per_race']} 円ずつ (賭け金の合計 {bet_total:,} 円)")
print(f"  予測 1 位に賭けた場合: 総利益 {top_pick['model_profit']:+,.0f} 円, 回収率 {top_pick['model_return']:.1%}")
print(f"  1 番人気に賭けた場合: 総利益 {top_pick['favorite_profit']:+,.0f} 円, 回収率 {top_pick['favorite_return']:.1%}")
print(f"  回収率の差 (予測 1 位 - 1 番人気): {top_pick['return_diff']:+.1%}  "
      f"(95% 信頼区間 {top_pick['return_diff_ci_low']:+.1%} 〜 {top_pick['return_diff_ci_high']:+.1%})")

display(pd.DataFrame({
    "レース数": [int(picks["same"].sum()), int((~picks["same"]).sum())],
    "予測1位が1着": [top_pick["hit_when_same"], top_pick["model_hit_when_diff"]],
    "1番人気が1着": [top_pick["hit_when_same"], top_pick["favorite_hit_when_diff"]],
}, index=["同じ馬", "違う馬"]).style.format({"予測1位が1着": "{:.1%}", "1番人気が1着": "{:.1%}"}))
print(f"片方だけが当たったレース: 予測 1 位だけ {top_pick['n_model_only']} / 1 番人気だけ {top_pick['n_favorite_only']}")

# 競馬場ごと (テストのレースが少ない競馬場はぶれが大きい)
if picks["race_venue"].nunique() > 1:
    picks["model_pay"] = picks["model_win"] * picks["model_odds"].fillna(0)  # 100 円あたりの払戻 / 100
    picks["fav_pay"] = picks["fav_win"] * picks["fav_odds"].fillna(0)
    by_venue = picks.groupby("race_venue").agg(
        レース数=("race_id", "size"), 予測1位が1着=("model_win", "mean"), 一番人気が1着=("fav_win", "mean"), 同じ馬の割合=("same", "mean"),
        回収率_予測1位=("model_pay", "mean"), 回収率_1番人気=("fav_pay", "mean"))
    by_venue["差"] = by_venue["予測1位が1着"] - by_venue["一番人気が1着"]
    display(by_venue.rename(columns={"一番人気が1着": "1番人気が1着", "回収率_予測1位": "回収率 予測1位", "回収率_1番人気": "回収率 1番人気"}).style.format(
        {"予測1位が1着": "{:.1%}", "1番人気が1着": "{:.1%}", "同じ馬の割合": "{:.1%}", "差": "{:+.1%}",
         "回収率 予測1位": "{:.1%}", "回収率 1番人気": "{:.1%}"}))


In [ ]:
# ROC 曲線 (左上に近いほど良い. 点線はランダム). Kaggle は日本語フォントがないため図中の文字は英語
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve

fig, ax = plt.subplots(figsize=(5, 5))
for label, data, col in [("Model", pred_test, "p_win"), ("Win odds", fav, "p_win")]:
    d = data.dropna(subset=[col])
    fpr, tpr, _ = roc_curve(d["is_win"].astype(int), d[col])
    auc = roc_auc_summary(d, col)["roc_auc"]
    ax.plot(fpr, tpr, label=f"{label} (AUC={auc:.3f})")
ax.plot([0, 1], [0, 1], linestyle=":", color="gray")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC curve: winner (test)")
ax.legend()
fig.savefig(RESULTS_DIR / "roc_curve.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# 予測した 1 着の確率と実際の 1 着率のずれ (mean_pred と actual_rate が近いほど良い)
calibration_table(pred_test)

### スコアと実際の 1 着率のずれ (頭数が同じになるように 10 等分)
スコア (`p_win`. 1 着の確率) の低い順に馬を並べ, **頭数が同じになるように 10 等分** する (1 = スコアが最も低い 10%, 10 = 最も高い 10%).
帯ごとに **スコアの平均** と **実際に 1 着になった割合** を比べる (単勝オッズから逆算した確率でも, それ自身の順で 10 等分して並べる).

- 範囲: その帯に入った馬のスコアの最小〜最大
- 差 = 実際の 1 着率 - スコアの平均. プラスならスコアが低すぎ (過小評価), マイナスなら高すぎ (過大評価). 0 に近いほど, スコアをそのまま確率として信じてよい
- 95% 信頼区間: 実際の 1 着率の偶然のぶれの幅. スコアの平均がこの外にあれば, 偶然では説明しにくいずれ
- ECE: 帯ごとの |差| を馬の数で重み付けした平均 (小さいほど良い)
- 等間隔 (0-10%, 10-20%, ...) で見たい場合は `compare_calibration(pred_test, method="width")`


In [ ]:
calibration = compare_calibration(pred_test)
cal_model = pd.DataFrame(calibration["model"]["bins"]).set_index("bin")
cal_odds = pd.DataFrame(calibration["odds"]["bins"]).set_index("bin")
cal_view = pd.concat({"モデル": cal_model[["range", "n", "mean_score", "win_rate", "diff", "ci_low", "ci_high"]],
                      "単勝オッズ": cal_odds[["range", "n", "mean_score", "win_rate", "diff"]]}, axis=1)
cal_view.columns = [f"{a} {b}" for a, b in cal_view.columns]
cal_view = cal_view.rename(columns=lambda c: c.replace("mean_score", "スコアの平均").replace("win_rate", "実際の1着率")
                           .replace("diff", "差").replace("ci_low", "1着率の95%下限").replace("ci_high", "1着率の95%上限").replace(" n", " 頭数").replace(" range", " 範囲"))
pct_cols = [c for c in cal_view.columns if "頭数" not in c and "範囲" not in c]
display(cal_view.style.format({c: "{:.1%}" for c in pct_cols}, na_rep="-").format({c: "{:,.0f}" for c in cal_view.columns if "頭数" in c}, na_rep="-"))
for name in ("model", "odds"):
    s = calibration[name]
    print(f"{'モデル' if name == 'model' else '単勝オッズ'}: ECE {s['ece']:.4f} / |差| の最大 (30 頭以上の帯) {s['max_abs_diff']:.1%} / "
          f"スコアの平均が 1 着率の 95% 信頼区間の外にある帯 {s['n_outside_ci']} 個")

# 信頼性曲線 (点線の対角線に近いほど良い). Kaggle は日本語フォントがないため図中の文字は英語
# 左: 普通の軸 / 右: 対数軸 (低い確率の帯のずれが見やすい)
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for ax, log in zip(axes, (False, True)):
    for label, t in [("Model", cal_model), ("Win odds", cal_odds)]:
        t = t[t["n"] > 0]
        ax.errorbar(t["mean_score"], t["win_rate"], yerr=[t["win_rate"] - t["ci_low"], t["ci_high"] - t["win_rate"]],
                    marker="o", capsize=3, label=label)
    lo = 0.001 if log else 0
    ax.plot([lo, 1], [lo, 1], linestyle=":", color="gray")
    if log:
        ax.set_xscale("log")
        ax.set_yscale("log")
    ax.set_xlabel("Mean score in decile (predicted win probability)")
    ax.set_ylabel("Actual win rate")
    ax.set_title("Score deciles (equal counts, test)" + (" - log scale" if log else ""))
    ax.legend()
fig.savefig(RESULTS_DIR / "calibration_bins.png", dpi=120, bbox_inches="tight")
plt.show()


### 特徴量の重要度
- `feature_ja`: 特徴量の日本語名 (`feature` は英語名)
  - 「今回の○○での平均着順」: 今回のレース条件 (芝・ダート / 距離 / 馬場状態 / 競馬場) と同じ条件での過去の平均着順
  - 「(補正後)」: 出走回数が少ないほど全体の平均着順に寄せた値 (1〜2 戦だけの成績でぶれないようにしたもの)
  - 「レース内順位」: 同じレースの出走馬の中での順位 (0〜1. 値が小さいほど小さい側) / 「レース内平均との差」: 同じレースの平均との差
  - 「スピード指数」: 走破タイムが, コース・距離・馬場状態ごとの基準タイムより 1000m あたり何秒速いか (大きいほど速い)
  - 「騎手の勝率」など: そのレースより前の学習データでの成績 (出走数が少ないほど全体の平均に寄せたもの)
  - 「着順/頭数」: 0 = 1 着, 1 = 最下位 / 「着差」: 秒 (1 着はマイナス)
- `gain`: その特徴量で分岐したときの損失の減少量の合計 (LightGBM 標準). `gain_share` は全体に占める割合
- `split`: 分岐に使われた回数 (多くても gain が小さければ効いていない)
- `shap`: テスト期間での SHAP 値の絶対値の平均. 予測を実際にどれだけ動かしたか (gain より解釈しやすい)

`IMPORTANCE_SORT_BY` で並び順の基準を選べる. 全特徴量を重要度の高い順に表示する.

In [ ]:
IMPORTANCE_SORT_BY = "shap"   # "shap" / "gain" / "split"

importance = feature_importance(booster, builder, test, sort_by=IMPORTANCE_SORT_BY)

with pd.option_context("display.max_rows", None):
    try:  # 棒付きの表 (jinja2 が必要. なければ普通の表で表示)
        display(importance.style.format({"gain": "{:.1f}", "gain_share": "{:.2%}", "shap": "{:.4f}"}).bar(subset=[IMPORTANCE_SORT_BY], color="#8ecae6"))
    except ImportError:
        display(importance)

In [ ]:
# 上位 30 特徴量の棒グラフ (重要度の高いものが上).
# 日本語フォントがあれば日本語名, なければ英語名で描く (フォントがない環境で日本語を描くと □ に文字化けするため)
SORT_LABEL_JA = {"shap": "SHAP 値の絶対値の平均", "gain": "gain (損失の減少量の合計)", "split": "分岐に使われた回数"}
top = importance.head(30).iloc[::-1]
fig, ax = plt.subplots(figsize=(8, 9))
if JAPANESE_FONT:
    ax.barh(top["feature_ja"], top[IMPORTANCE_SORT_BY])
    ax.set_xlabel(SORT_LABEL_JA.get(IMPORTANCE_SORT_BY, IMPORTANCE_SORT_BY))
    ax.set_title(f"特徴量の重要度 (上位 30, テスト期間)")
else:
    ax.barh(top["feature"], top[IMPORTANCE_SORT_BY])
    ax.set_xlabel(IMPORTANCE_SORT_BY)
    ax.set_title(f"Feature importance (top 30, {IMPORTANCE_SORT_BY})")
plt.tight_layout()
fig.savefig(RESULTS_DIR / "feature_importance_top30.png", dpi=120, bbox_inches="tight")
plt.show()

# 使われていない特徴量 (分岐に一度も使われなかったもの)
unused = importance.loc[importance["split"] == 0, "feature_ja"].tolist()
print(f"分岐に使われなかった特徴量: {len(unused)} / {len(importance)}")
print("  " + " / ".join(unused))

### 評価結果の保存
評価結果をファイルに保存する (Kaggle では `/kaggle/working/results/`, ローカルでは `reports/latest/`).
`scripts/train_on_kaggle.py` で実行した場合は, ローカルの `reports/kaggle_runs/<日時>/results/` にダウンロードされる.

| ファイル | 内容 |
|---|---|
| `summary.json` | 設定, 学習・テスト期間, CV / テストの評価指標 (モデルと単勝オッズ), 最良パラメータ |
| `test_predictions.csv` | テスト期間の各馬の予測確率と結果 |
| `calibration.csv` | 予測確率帯ごとの実際の 1 着率 |
| `feature_importance_test.csv` | 特徴量の重要度 (gain / split / SHAP) |
| `optuna_trials.csv` | Optuna の全試行のパラメータとスコア |
| `roc_curve.png` / `calibration_bins.png` / `feature_importance_top30.png` | グラフ |

In [ ]:
import json


def _to_builtin(o):
    """numpy の数値などを JSON に書ける形にする."""
    return o.item() if hasattr(o, "item") else str(o)


summary = {
    "config": cfg.to_dict(),
    "data_period": [str(df["race_date"].min().date()), str(df["race_date"].max().date())],
    "train_period": [str(train["race_date"].min().date()), str(train["race_date"].max().date())],
    "test_period": [str(test["race_date"].min().date()), str(test["race_date"].max().date())],
    "n_races": {"train": int(train["race_id"].nunique()), "test": int(test["race_id"].nunique())},
    "cv_logloss": result.best_score,
    "num_boost_round": result.num_boost_round,
    "best_params": result.best_params,
    "test_metrics": {"model": metrics, "odds_baseline": baseline},
    "market_offset": MARKET_OFFSET,
    "test_market_blend": market_blend,
    "test_top_pick": top_pick,
    "test_calibration": calibration,
    "n_unused_features": len(unused),
}
(RESULTS_DIR / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2, default=_to_builtin), encoding="utf-8")

prediction_cols = [
    "race_id", "race_date", "race_venue", "umaban", "horse_name",
    "win_score", "p_win", "pred_rank", "rank", "is_win", "is_top3", "popularity", "win_odds",
]
pred_test[prediction_cols].to_csv(RESULTS_DIR / "test_predictions.csv", index=False, encoding="utf-8-sig")
calibration_table(pred_test).astype({"p_win_bin": str}).to_csv(RESULTS_DIR / "calibration.csv", index=False, encoding="utf-8-sig")
importance.to_csv(RESULTS_DIR / "feature_importance_test.csv", encoding="utf-8-sig")
result.study.trials_dataframe().to_csv(RESULTS_DIR / "optuna_trials.csv", index=False, encoding="utf-8-sig")

print("評価結果を保存しました:", RESULTS_DIR)
for p in sorted(RESULTS_DIR.iterdir()):
    print(f"  {p.name} ({p.stat().st_size:,} bytes)")

## 5. 全期間で学習し直して保存
テストで性能を確認したら, テスト期間も含めた全期間 (`data.start_date` 〜 `data.end_date`) で学習し直したモデルを推論用に保存する.
Kaggle では `/kaggle/working/artifacts` に保存される (ノートブックの Output からダウンロードできる).

In [ ]:
builder_full, booster_full = train_final(df, result.best_params, result.num_boost_round, market_offset=MARKET_OFFSET)
save_artifacts(
    ARTIFACTS_DIR,
    booster_full,
    builder_full,
    result.best_params,
    result.num_boost_round,
    train=df,
    metrics={
        "test": metrics,
        "test_odds_baseline": baseline,
        "test_market_blend": market_blend,
        "test_top_pick": top_pick,
        "test_calibration": calibration,
        "cv_logloss": result.best_score,
        "test_ratio": TEST_RATIO,
        "config": cfg.to_dict(),
        "test_period": [str(test["race_date"].min().date()), str(test["race_date"].max().date())],
    },
)
feature_importance(booster_full, builder_full, df).to_csv(ARTIFACTS_DIR / "feature_importance.csv", encoding="utf-8-sig")
print("保存しました:", ARTIFACTS_DIR, [p.name for p in ARTIFACTS_DIR.iterdir()])

## 6. 推論
保存したモデルを読み込み, レースごとに 1 着の確率と, そこからシミュレーションした 3 着以内・ワイドの確率を出す (テスト期間のレースで例示).

In [ ]:
booster_loaded, builder_loaded, metadata = load_artifacts(ARTIFACTS_DIR)

race_id = test["race_id"].iloc[-1]   # 予測したいレース
race = predict_win(test[test["race_id"] == race_id], booster_loaded, builder_loaded).sort_values("pred_rank").reset_index(drop=True)
in_top = simulate_top3(race)   # 1 着の確率を各馬の強さとして着順をシミュレーション
race["p_top3"] = in_top.mean(axis=0)
print("p_win の合計:", race["p_win"].sum())
display(race[["umaban", "horse_name", "p_win", "p_top3", "pred_rank", "rank", "popularity", "win_odds"]])

# ワイドの的中確率 (上位 10 組)
wide_probabilities(race, in_top=in_top).head(10)

### これから行われるレースを予測する (任意)
netkeiba から出馬表・戦績・血統を取得して予測する. 馬体重の発表 (発走 60 分前) 後に実行する.
オッズとの差を学習したモデルでは, 今の単勝オッズも取得して使う (発売前は予測できない).
Kaggle で実行する場合はノートブックの設定で **Internet を ON** にする. 1 レースあたり 1〜2 分かかる.

In [ ]:
from racing_prediction.data.make_dataset import build_race_table

UPCOMING_RACE_ID = None   # 例: "202605050611"

if UPCOMING_RACE_ID:
    upcoming = build_race_table(UPCOMING_RACE_ID)
    if builder_loaded.market_offset:  # オッズとの差を学習したモデルは, 今の単勝オッズが必要
        from racing_prediction.inference.run import apply_live_win_odds
        upcoming, odds_info = apply_live_win_odds(upcoming, UPCOMING_RACE_ID)
        print("単勝オッズ:", odds_info["win_odds_note"])
    upcoming_pred = predict_win(upcoming, booster_loaded, builder_loaded).sort_values("pred_rank").reset_index(drop=True)
    upcoming_pred["p_top3"] = simulate_top3(upcoming_pred).mean(axis=0)
    display(upcoming_pred[["umaban", "horse_name", "horse_weight", "weight_diff", "p_win", "p_top3", "pred_rank"]])
    display(wide_probabilities(upcoming_pred).head(10))